# ทำ reply.ipynb ให้เจน 3 แบบ: Base / Emotion-Enhanced / Dissonance-Based

## โหลดข้อมูล + เตรียมตาราง

In [1]:
import os
import json
import pandas as pd
from openai import OpenAI

BASE_DIR = "/home/patsakornt/work/test"
OWN_SCRIPT_DIR = os.path.join(BASE_DIR, "dissonance", "own_script", "dialogue_2")

# 1) โหลด dissonance (จาก df_dissonance ที่เซฟไว้)
diss_path = os.path.join(OWN_SCRIPT_DIR, "dialogue_2_dissonance.csv")
df_diss = pd.read_csv(diss_path)

print(df_diss)

   utterance_id     aro_s     val_s     aro_t     val_t  delta_arousal  \
0             1  0.155329 -0.674792  0.248168 -0.242050       0.092839   
1             2 -0.080474  0.001581  0.286293 -0.248449       0.366767   
2             3  0.735761  0.534915  0.427576  0.519725       0.308185   
3             4 -0.241028  0.075200  0.131011  0.173824       0.372038   

   delta_valence  dissonant_arousal  dissonant_valence  dissonant_any  
0       0.432742              False              False          False  
1       0.250030              False              False          False  
2       0.015191              False              False          False  
3       0.098624              False              False          False  


## ต่อไปเอา text มาติดด้วย

In [2]:
# ถ้าคุณมี dialogue_2_vad_text.csv พร้อม text อยู่แล้ว:
text_vad_path = os.path.join(OWN_SCRIPT_DIR, "dialogue_2_vad_text.csv")
df_text = pd.read_csv(text_vad_path)

# เอาเฉพาะ utterance_id + text
df_text = df_text[["utterance_id", "text"]]

# merge เข้ากับ df_diss
df = pd.merge(df_diss, df_text, on="utterance_id", how="left")

df = df.sort_values("utterance_id").reset_index(drop=True)
df


,utterance_id,aro_s,val_s,aro_t,val_t,delta_arousal,delta_valence,dissonant_arousal,dissonant_valence,dissonant_any,text
0,1,0.155329,-0.674792,0.248168,-0.242050,0.092839,0.432742,False,False,False,Why are you bothering me? What's the problem?
1,2,-0.080474,0.001581,0.286293,-0.248449,0.366767,0.250030,False,False,False,"Ahh that thing again, can you just stay away f..."
2,3,0.735761,0.534915,0.427576,0.519725,0.308185,0.015191,False,False,False,I'm fine! I am very good and doing well at the...
3,4,-0.241028,0.075200,0.131011,0.173824,0.372038,0.098624,False,False,False,"Besides, you are the one who seems to be doing..."


### ตั้ง OpenAI client

In [4]:
import getpass
import os

print("Setting up OpenAI client...")
if "OPENAI_API_KEY" not in os.environ:
    secret_key = getpass.getpass("Enter your OpenAI API key: ")
    os.environ["OPENAI_API_KEY"] = secret_key

client = OpenAI()
OPENAI_MODEL_ID = "gpt-4o-mini"
print("✅ OpenAI client ready:", OPENAI_MODEL_ID)


Setting up OpenAI client...
✅ OpenAI client ready: gpt-4o-mini


## Prompt templates ทั้ง 3 แบบ

### System prompt ร่วม (Luna Therapist)

In [5]:
SYSTEM_PROMPT = """You are "Luna", an empathetic CBT-oriented AI therapist. 
You respond warmly, validate emotions, and use gentle CBT strategies 
(reflection, thought-challenging, and collaborative planning).

Your answers must:
- Be 2–4 sentences.
- Be concrete and supportive.
- End with one open-ended question.
"""


### Base prompting (text only)

In [6]:
BASE_USER_TEMPLATE = """The client just said:

"{text}"

As Luna, please respond directly to the client.
Do NOT mention that you are an AI or that you saw any extra data.
"""


### Emotion-Enhanced prompting (ใช้ VA แต่ไม่พูดถึง dissonance)

In [7]:
EMO_USER_TEMPLATE = """The client just said:

"{text}"

We also estimated the client's emotional state on a valence-arousal scale:
- Text-based estimate: valence={val_t:.3f}, arousal={aro_t:.3f}
- Voice-based estimate: valence={val_s:.3f}, arousal={aro_s:.3f}

Use this information implicitly to guide your tone and content
(e.g., acknowledge high arousal or low valence),
but do NOT explicitly mention numbers or the measurement process.
Respond as Luna directly to the client.
"""


### Dissonance-Based prompting (เน้น mismatch)

In [8]:
DIS_USER_TEMPLATE = """The client just said:

"{text}"

We estimated the client's emotional state on a valence-arousal scale:
- Text-based estimate: valence={val_t:.3f}, arousal={aro_t:.3f}
- Voice-based estimate: valence={val_s:.3f}, arousal={aro_s:.3f}

There is an emotional dissonance between what the client says in words
and how they sound in their voice:
- Difference in arousal = {delta_arousal:.3f}
- Difference in valence = {delta_valence:.3f}
Dissonance flag = {dissonant_any}

Assume this mismatch might mean the client is downplaying or masking 
some feelings, or that their voice conveys more intensity than their words.

In your reply as Luna:
- Gently acknowledge both the content and the possible hidden/emotional layer.
- Validate the feelings that might not be fully stated.
- If appropriate, check for misunderstanding or minimization.
- Then continue with a brief CBT-style exploration.

Do NOT mention the words "dissonance", "valence", "arousal", or any numbers.
Speak naturally as a therapist to the client.
"""


## Loop รันทั้ง 3 conditions สำหรับ dialogue_2

In [9]:
results = []

for _, row in df.iterrows():
    utt_id = int(row["utterance_id"])
    text = row["text"]

    aro_s = float(row["aro_s"])
    val_s = float(row["val_s"])
    aro_t = float(row["aro_t"])
    val_t = float(row["val_t"])
    delta_a = float(row["delta_arousal"])
    delta_v = float(row["delta_valence"])
    diss_any = bool(row["dissonant_any"])

    # 1) Base
    base_user = BASE_USER_TEMPLATE.format(text=text)
    base_msgs = [
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user", "content": base_user},
    ]
    try:
        base_completion = client.chat.completions.create(
            model=OPENAI_MODEL_ID,
            messages=base_msgs,
            max_tokens=256,
            temperature=0.6,
            top_p=0.9,
        )
        base_reply = base_completion.choices[0].message.content.strip()
    except Exception as e:
        base_reply = f"ERROR: {e}"

    # 2) Emotion-Enhanced
    emo_user = EMO_USER_TEMPLATE.format(
        text=text,
        val_t=val_t,
        aro_t=aro_t,
        val_s=val_s,
        aro_s=aro_s,
    )
    emo_msgs = [
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user", "content": emo_user},
    ]
    try:
        emo_completion = client.chat.completions.create(
            model=OPENAI_MODEL_ID,
            messages=emo_msgs,
            max_tokens=256,
            temperature=0.6,
            top_p=0.9,
        )
        emo_reply = emo_completion.choices[0].message.content.strip()
    except Exception as e:
        emo_reply = f"ERROR: {e}"

    # 3) Dissonance-Based
    dis_user = DIS_USER_TEMPLATE.format(
        text=text,
        val_t=val_t,
        aro_t=aro_t,
        val_s=val_s,
        aro_s=aro_s,
        delta_arousal=delta_a,
        delta_valence=delta_v,
        dissonant_any=diss_any,
    )
    dis_msgs = [
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user", "content": dis_user},
    ]
    try:
        dis_completion = client.chat.completions.create(
            model=OPENAI_MODEL_ID,
            messages=dis_msgs,
            max_tokens=256,
            temperature=0.6,
            top_p=0.9,
        )
        dis_reply = dis_completion.choices[0].message.content.strip()
    except Exception as e:
        dis_reply = f"ERROR: {e}"

    results.append({
        "utterance_id": utt_id,
        "client_text": text,
        "aro_s": aro_s,
        "val_s": val_s,
        "aro_t": aro_t,
        "val_t": val_t,
        "delta_arousal": delta_a,
        "delta_valence": delta_v,
        "dissonant_any": diss_any,
        "reply_base": base_reply,
        "reply_emotion": emo_reply,
        "reply_dissonance": dis_reply,
    })

len(results)


4

## เซฟเป็น CSV / JSONL ไว้ใช้ต่อ

In [10]:
out_df = pd.DataFrame(results)
out_csv = os.path.join(OWN_SCRIPT_DIR, "dialogue_2_replies_all_conditions.csv")
out_df.to_csv(out_csv, index=False)
print("Saved:", out_csv)

# เผื่ออยากเก็บเป็น JSONL สำหรับ future training
out_jsonl = os.path.join(OWN_SCRIPT_DIR, "dialogue_2_replies_all_conditions.jsonl")
with open(out_jsonl, "w", encoding="utf-8") as f:
    for rec in results:
        f.write(json.dumps(rec, ensure_ascii=False) + "\n")
print("Saved:", out_jsonl)


Saved: /home/patsakornt/work/test/dissonance/own_script/dialogue_2/dialogue_2_replies_all_conditions.csv
Saved: /home/patsakornt/work/test/dissonance/own_script/dialogue_2/dialogue_2_replies_all_conditions.jsonl
